# 02 - Data Preprocessing

**Project:** DetectIt – Intelligent Web Threat Detection Using Machine Learning

**Goal:** In the EDA notebook, we found a few problems in the dataset and made a plan to fix them. In this notebook, we follow that plan step by step. At the end, we save one training set and one test set, so every team member can train their models on the same data.

**Input:** `data/PhiUSIIL_Phishing_URL_Dataset.csv`

**Output:** `data/processed/train.csv`, `data/processed/test.csv`, `data/processed/feature_names.json`

## Setup

We load the libraries and keep all settings in one place. The columns we remove later were chosen in the EDA notebook.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "label"
TEXT_COLS = ["FILENAME", "URL", "Domain", "TLD", "Title"]
REDUNDANT_COLS = ["URLTitleMatchScore", "URLLength"]

In [2]:
df = pd.read_csv(DATA_PATH)
initial_shape = df.shape
print(f"Loaded dataset: {initial_shape[0]:,} rows | {initial_shape[1]} columns")

Loaded dataset: 235,795 rows | 56 columns


## 1. Data Validation

Before we change anything, we check that the values make sense. A count, like the number of images on a page, can never be negative. A yes/no column should only have 0 or 1. The label should also only be 0 or 1.

In [3]:
numeric_cols = df.select_dtypes(include="number").columns.drop(TARGET)
binary_cols = [col for col in numeric_cols if df[col].nunique() == 2]

negative_cols = [col for col in numeric_cols if (df[col] < 0).any()]
invalid_binary_cols = [col for col in binary_cols if not set(df[col].unique()) <= {0, 1}]
invalid_labels = set(df[TARGET].unique()) - {0, 1}

print(f"Columns with negative values: {negative_cols}")
print(f"Binary columns with values other than 0/1: {invalid_binary_cols}")
print(f"Unexpected label values: {invalid_labels}")

Columns with negative values: []
Binary columns with values other than 0/1: []
Unexpected label values: set()


## 2. Remove Duplicate URLs

In the EDA, we found 425 URLs that appear twice. If we keep them, one copy could go into the training set and the other into the test set. The model would then be tested on a URL it has already seen, and the results would look better than they really are. So we keep the first copy of each URL and remove the second.

In [4]:
duplicate_count = df["URL"].duplicated().sum()
df = df.drop_duplicates(subset="URL", keep="first").reset_index(drop=True)

print(f"Removed {duplicate_count} duplicate URLs")
print(f"Rows remaining: {len(df):,}")

Removed 425 duplicate URLs
Rows remaining: 235,370
